In [66]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# from transformers import pipeline
# import torch
from datasets import load_dataset
import pandas as pd
import numpy as np
import json

In [67]:
ds = load_dataset("data-is-better-together/10k_prompts_ranked")
ds

DatasetDict({
    train: Dataset({
        features: ['prompt', 'quality', 'metadata', 'avg_rating', 'num_responses', 'agreement_ratio', 'raw_responses', 'kind', 'cluster_description', 'topic'],
        num_rows: 10331
    })
})

In [68]:
train_data = ds['train']

In [69]:
train_data

Dataset({
    features: ['prompt', 'quality', 'metadata', 'avg_rating', 'num_responses', 'agreement_ratio', 'raw_responses', 'kind', 'cluster_description', 'topic'],
    num_rows: 10331
})

In [70]:
cnt = 0
for sample in train_data:
  if cnt != 3:
    print(sample['prompt'], end = "\n")
    cnt += 1

Provide step-by-step instructions on how to make a safe and effective homemade all-purpose cleaner from common household ingredients. The guide should include measurements, tips for storing the cleaner, and additional variations or scents that can be added. Additionally, the guide should be written in clear and concise language, with helpful visuals or photographs to aid in the process.
Write a personal essay of at least 1000 words discussing how embracing vulnerability and authenticity has affected your life. Use specific examples from your own experiences to support your arguments and make sure to address the following questions:
In this research, we aim to investigate how technology can moderate the correlation between knowledge management practices and the overall performance of an organization. This analysis will focus on specific technological tools and their impact on knowledge management in relation to various aspects of organizational performance. Additionally, we will explore

In [71]:
def createDataFrame(ds) -> pd.DataFrame:
  train_data = ds['train']
  data_label = []
  for sample in train_data:
    data_label.append(
        {
            "text" : sample["prompt"],
            "quality" : sample["quality"][0]['value'],
            "avg_rating" : sample["avg_rating"],
            "kind" : sample["kind"]
        }
    )
  df = pd.DataFrame(data_label)
  return df

In [72]:
def is_need(row):
  if (row['quality'] < 3) and (row['avg_rating'] < 2.5) :
    return "not good"
  if (row['quality'] < 3) and (row['avg_rating'] > 2.5 and row['avg_rating'] < 3.5):
    return "mid"
  return "good"
def incomplete(row):
  if row['quality'] < 3.0 and row['avg_rating'] < 2.7:
    return "reproduce"
  return "complete"
df = createDataFrame(ds)
df['quality'] = df['quality'].astype('float64')
gl_label = []
inc = []
for i in range(len(df)):
  gl_label.append(is_need(df.iloc[i]))
  inc.append(incomplete(df.iloc[i]))
df['gl_label'] = gl_label
df['inc_label'] = inc

In [73]:
df.head()

,text,quality,avg_rating,kind,gl_label,inc_label
0,Provide step-by-step instructions on how to ma...,4.0,5.000000,synthetic,good,complete
1,Write a personal essay of at least 1000 words ...,1.0,2.750000,synthetic,mid,complete
2,"In this research, we aim to investigate how te...",2.0,3.000000,synthetic,mid,complete
3,Did Karl Marx's theories on centralizing credi...,3.0,3.500000,human,good,complete
4,"alter this api that gets a request like: {""0"",...",4.0,3.666667,human,good,complete


In [74]:
df.inc_label.value_counts()

,count
inc_label,
complete,8966
reproduce,1365


In [154]:
#df.to_csv("draft_1.csv", index = False)

In [76]:
def to_binary(x):
    return 1 if x > 0.5 else 0
def cFrame(ds) -> pd.DataFrame:
  train_data = ds['train']
  data_label = []
  for sample in train_data:
      text = sample["prompt"]["text"]
      data_label.append({
          "text": text,
          "toxicity": sample["prompt"]["toxicity"],
          "insult": sample["prompt"]["insult"],
          "threat": sample["prompt"]["threat"],
          "identity_attack": sample["prompt"]["identity_attack"],
          "profanity": sample["prompt"]["profanity"]
      })

  df = pd.DataFrame(data_label)
  df['toxic_label'] = df['toxicity'].apply(to_binary)
  df['insult_label'] = df['insult'].apply(to_binary)
  df['threat_label'] = df['threat'].apply(to_binary)
  df['identity_attack_label'] = df['identity_attack'].apply(to_binary)
  df['profanity_label'] = df['profanity'].apply(to_binary)
  return df

ds = load_dataset("allenai/real-toxicity-prompts")
df2 = cFrame(ds)

In [77]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99442 entries, 0 to 99441
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   text                   99442 non-null  object 
 1   toxicity               99016 non-null  float64
 2   insult                 99016 non-null  float64
 3   threat                 99016 non-null  float64
 4   identity_attack        99016 non-null  float64
 5   profanity              99016 non-null  float64
 6   toxic_label            99442 non-null  int64  
 7   insult_label           99442 non-null  int64  
 8   threat_label           99442 non-null  int64  
 9   identity_attack_label  99442 non-null  int64  
 10  profanity_label        99442 non-null  int64  
dtypes: float64(5), int64(5), object(1)
memory usage: 8.3+ MB


In [144]:
df1 = df2.copy()
df1 = df1.dropna()
# df_new = pd.DataFrame()
# new_data = pd.DataFrame(df1[(df1['toxic_label'] == 1) & (~df1['toxic_label'].isna())][:600])
# df_new = pd.concat([df_new, new_data], ignore_index = True)
# df1 = df1.drop(new_data.index)
# new_data = pd.DataFrame(df1[(df1['toxic_label'] == 1) & (df1['profanity_label'] == 1)][:700])
# df_new = pd.concat([df_new, new_data], ignore_index = True)
# df1 = df1.drop(new_data.index)
# new_data = pd.DataFrame(df1[(df1['toxic_label'] == 1) & (df1['threat_label'] == 1)][:600])
# df_new = pd.concat([df_new, new_data], ignore_index = True)
# df1 = df1.drop(new_data.index)
# new_data = pd.DataFrame(df1[(df1['toxic_label'] == 1) & (df1['profanity_label'] == 1) & (df1['threat_label'] == 1)][:700])
# df_new = pd.concat([df_new, new_data], ignore_index = True)
# df1 = df1.drop(new_data.index)

In [145]:
df_new = pd.DataFrame()

def move_data(condition, count):
    global df1, df_new
    subset = df1[condition].head(count)
    df_new = pd.concat([df_new, subset], ignore_index=True)
    df1 = df1.drop(subset.index)
    print(f"Moved {len(subset)} rows. Remaining in df1: {len(df1)}")

df1 = df1.dropna()
move_data((df1['toxic_label'] == 1) & (df1['profanity_label'] == 0) & (df1['threat_label'] == 0), 900) #Toxic Only
# Profanity only
move_data((df1['toxic_label'] == 0) & (df1['profanity_label'] == 1) & (df1['threat_label'] == 0), 900)
# Threat only
move_data((df1['toxic_label'] == 0) & (df1['profanity_label'] == 0) & (df1['threat_label'] == 1), 900)
# Toxic + Profane (Not Threat)
move_data((df1['toxic_label'] == 1) & (df1['profanity_label'] == 1) & (df1['threat_label'] == 0), 900)
# Profane + Threat (Not Toxic)
move_data((df1['toxic_label'] == 0) & (df1['profanity_label'] == 1) & (df1['threat_label'] == 1), 900)
# Toxic + Threat (Not Profane)
move_data((df1['toxic_label'] == 1) & (df1['profanity_label'] == 0) & (df1['threat_label'] == 1), 900)
# All Three
move_data((df1['toxic_label'] == 1) & (df1['profanity_label'] == 1) & (df1['threat_label'] == 1), 1400)

Moved 900 rows. Remaining in df1: 98116
Moved 169 rows. Remaining in df1: 97947
Moved 900 rows. Remaining in df1: 97047
Moved 900 rows. Remaining in df1: 96147
Moved 9 rows. Remaining in df1: 96138
Moved 900 rows. Remaining in df1: 95238
Moved 1400 rows. Remaining in df1: 93838


In [146]:
#df = pd.concat([df, df_new], ignore_index = True)
dl = df.copy()
dl = dl.sample(frac=0.85, random_state=42)

In [147]:
dl = pd.concat([dl, df_new], ignore_index = True)

In [148]:
dl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13959 entries, 0 to 13958
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   text                   13959 non-null  object 
 1   quality                8781 non-null   float64
 2   avg_rating             8781 non-null   float64
 3   kind                   8781 non-null   object 
 4   gl_label               8781 non-null   object 
 5   inc_label              8781 non-null   object 
 6   toxicity               5178 non-null   float64
 7   insult                 5178 non-null   float64
 8   threat                 5178 non-null   float64
 9   identity_attack        5178 non-null   float64
 10  profanity              5178 non-null   float64
 11  toxic_label            5178 non-null   float64
 12  insult_label           5178 non-null   float64
 13  threat_label           5178 non-null   float64
 14  identity_attack_label  5178 non-null   float64
 15  pr

In [149]:
dl = dl.drop(columns = ['insult','insult_label','identity_attack_label','identity_attack'],axis = 1)

In [150]:
dl['quality'] = dl['quality'].astype('float64').fillna(0)
dl['avg_rating'] = dl['avg_rating'].astype('float64').fillna(0)
dl['gl_label'] = dl['gl_label'].fillna("not good")
dl['inc_label'] = dl['inc_label'].fillna("reproduce")
kind_nuls = dl['kind'].isnull()
num_nulls = kind_nuls.sum()
random_values = np.random.choice(["synthetic","human"], size=num_nulls)
dl.loc[kind_nuls, 'kind'] = random_values
dl['toxic_label'] = dl['toxic_label'].astype('float64').fillna(0)
dl['threat_label'] = dl['threat_label'].astype('float64').fillna(0)
dl['profanity_label'] = dl['profanity_label'].astype('float64').fillna(0)

cols_to_fix = ['toxicity', 'profanity', 'threat']
for col in cols_to_fix:
    is_null = dl[col].isnull()
    num_nulls = is_null.sum()
    # Generate random "safe" values between 0.0001 and 0.01
    # You can adjust high=0.01 to whatever "very low" means to you
    low_noise = np.random.uniform(low=0.0001, high=0.01, size=num_nulls)
    dl.loc[is_null, col] = low_noise
print("Nulls filled with low-intensity noise.")

Nulls filled with low-intensity noise.


In [151]:
dl.isnull().sum()

,0
text,0
quality,0
avg_rating,0
kind,0
gl_label,0
inc_label,0
toxicity,0
threat,0
profanity,0
toxic_label,0


In [153]:
dl.to_csv("draft_2.csv", index = False)